# 🤖 Find your Copilot Studio robots with Python

(explained super simply)

What are we going to do? We will ask Microsoft: "Please show me all the Copilot Studio agents I am allowed to see." Then we will find your own agent.

You do not need to understand everything. Just:
1. Read the little story above each step.
2. Click the cell and press Shift + Enter to run it.
3. Go to the next step.

💡 Only one thing may need changing: your email in Step 1.

## 👧 The story

Imagine a giant toy building 🏢.

* Inside the building there are many rooms 🚪. In Microsoft these rooms are called environments.
* In each room there is a shelf 📚 that lists every toy robot 🤖 in that room. The shelf is called Dataverse.
* Each toy robot is a Copilot Studio agent (a helper that can chat and do small jobs).
* To walk in, the guard checks who you are and gives you a visitor sticker 🏷️. That sticker is called a token. You get it when you sign in (the browser opens and you type your password / approve on your phone).
* Some rooms are locked 🔒 for you. The computer says 403 Forbidden. That is normal: you simply don't have a key for that room.

```
YOU  --sign in-->  guard gives you a sticker (token)
                         |
                         v
              ask: "which rooms are there?"
                         |
       +-----------------+-----------------+
       |                 |                 |
    room 1 🚪         room 2 🚪         room 3 🚪 ... (environments)
       |                 |                 |
     shelf 📚          shelf 📚         🔒 (locked, 403)
    🤖 🤖 🤖          🤖 🤖
```

So Python does exactly this: sign in -> list rooms -> read the shelf in each room -> print robot names.

## 📖 Little dictionary

| Big word | What it really means |
| :--- | :--- |
| **Agent** | A robot helper you build in Copilot Studio |
| **Environment** | A room where agents live |
| **Dataverse** | The shelf in the room that lists the agents |
| **Token** | A visitor sticker that proves who you are |
| **SSO / MFA** | Signing in with your work account, then approving on your phone |
| **403** | "This room is locked for you" |
| **Draft** | A robot that is built but not shared yet (never published) |

## Step 0️⃣ Get our tools ready

If you see an error like No module named msal, remove the # in the first line and run the cell once.

In [ ]:
# !pip install msal requests    # <- remove the '#' only if you get "No module named ..."
import json
import os
from concurrent.futures import ThreadPoolExecutor

import msal
import requests

print("Tools are ready 🟩")

## Step 1️⃣ Tell Python who you are

Only change USER if your work email is different. The other lines are fixed Microsoft addresses. Leave them alone.

* CLIENT_ID = the name of the "door" we use to sign in (a public Microsoft one, nothing secret).
* BAP_... = the address where Microsoft keeps the list of rooms.
* BOT_SELECT = which facts we want about each robot (name, id, dates...).
* CACHE_FILE = a small file where we keep your sticker so you don't need to sign in again every time.

In [ ]:
USER = ""    # <- your work email

CLIENT_ID = "51f81489-12ee-4a9e-aaae-a2591f45987d"
TENANT = "organizations"

BAP_SCOPE = "https://api.bap.microsoft.com/.default"
BAP_ENVS = ("https://api.bap.microsoft.com/providers/Microsoft.BusinessAppPlatform"
            "/environments?api-version=2020-10-01")
BOT_SELECT = "botid,name,schemaname,statecode,createdon,modifiedon,publishedon,language"
CACHE_FILE = ".token_cache.json"

print("Hello,", USER)

## Step 2️⃣ Sign in and get the visitor sticker 🏷️

When you run the next cell, a browser window opens. Sign in and approve on your phone (MFA). That's the only "hard" moment. After this, Python remembers you.

The function get_token("some address") means: "Guard, please give me a sticker for this address." (Every address needs its own sticker.)

⚠️ The file .token_cache.json is like a password. Never share it or put it on GitHub.

In [ ]:
cache = msal.SerializableTokenCache()
if os.path.exists(CACHE_FILE):
    cache.deserialize(open(CACHE_FILE).read())

app = msal.PublicClientApplication(
    CLIENT_ID, authority=f"https://login.microsoftonline.com/{TENANT}", token_cache=cache
)

def get_token(scope):
    # 1) Do I already have a sticker? Then use it (no browser).
    accounts = app.get_accounts(username=USER)
    if accounts:
        result = app.acquire_token_silent([scope], account=accounts[0])
        if result and "access_token" in result:
            return result["access_token"]
    # 2) No sticker yet? Open the browser so the guard can check me.
    result = app.acquire_token_interactive(scopes=[scope], login_hint=USER)
    if "access_token" not in result:
        raise RuntimeError(result.get("error_description", result))
    return result["access_token"]

def save_cache():
    if cache.has_state_changed:
        open(CACHE_FILE, "w").write(cache.serialize())
        os.chmod(CACHE_FILE, 0o600)

def get_all(url, token, next_key):
    # Microsoft hands out answers one PAGE at a time. This keeps turning pages until done.
    headers = {"Authorization": f"Bearer {token}", "Accept": "application/json"}
    items = []
    while url:
        r = requests.get(url, headers=headers, timeout=30)
        r.raise_for_status()
        data = r.json()
        items += data.get("value", [])
        url = data.get(next_key)
    return items

print("Sign-in helpers are ready 🟩 (the browser opens in the next step)")

## Step 3️⃣ Ask: "Which rooms are in the building?" 🚪

We use the sticker to ask Microsoft for all environments (rooms). We keep only rooms that have a shelf (Dataverse). A room without a shelf cannot hold robots.

👉 Your browser opens now. Sign in and approve MFA.

In [ ]:
bap_token = get_token(BAP_SCOPE)

envs = []
for e in get_all(BAP_ENVS, bap_token, "nextLink"):
    props = e.get("properties", {})
    shelf_url = (props.get("linkedEnvironmentMetadata") or {}).get("instanceUrl")
    if shelf_url:
        envs.append({"name": props.get("displayName", e["name"]), "url": shelf_url.rstrip("/")})

save_cache()
print(f"I found {len(envs)} rooms that have a shelf. First 5:")
for e in envs[:5]:
    print("  🚪", e["name"])

## Step 4️⃣ Look at the shelf in every room 📚

For each room:
1. Get a sticker for that room.
2. Read the bots shelf (that is where the robots are listed).
3. If the room is locked 🔒 (403), write it down as skipped and move on.

This can take a minute or two because there are many rooms.

In [ ]:
# Get one sticker per room (one by one, so we never open many browser windows at once)
tokens = {}
for env in envs:
    try:
        tokens[env["url"]] = get_token(env["url"] + "/.default")
    except Exception as exc:
        env["error"] = str(exc)
save_cache()

def read_shelf(env):
    token = tokens.get(env["url"])
    if not token:
        return env, None, env.get("error", "no sticker")
    try:
        bots = get_all(f"{env['url']}/api/data/v9.2/bots?$select={BOT_SELECT}", token, "@odata.nextLink")
        return env, bots, None
    except requests.HTTPError as exc:
        code = exc.response.status_code
        return env, None, "Locked room (403)" if code == 403 else f"error {code}"

with ThreadPoolExecutor(max_workers=8) as pool:    # ask many rooms at the same time = faster
    results = list(pool.map(read_shelf, envs))

agents, locked_rooms = [], []
for env, bots, problem in results:
    if problem:
        locked_rooms.append({"room": env["name"], "url": env["url"], "why": problem})
        continue
    for b in bots:
        agents.append({
            "room": env["name"],
            "room_url": env["url"],
            "name": b.get("name"),
            "agent_id": b["botid"],
            "published_on": b.get("publishedon"),    # empty = draft
        })

print(f"🤖 Robots (agents) found: {len(agents)}")
print(f"🔒 Locked rooms I could not open: {len(locked_rooms)}")

## Step 5️⃣ How many robots are in each room? 📊

In [ ]:
counts = {}
for a in agents:
    counts[a["room"]] = counts.get(a["room"], 0) + 1

for room, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"{n:5}  🤖 in {room}")

## Step 6️⃣ Print ALL the robot names 📜

This list is long (it is a big building!). Scroll the output, or use the saved file in Step 8.

In [ ]:
names = sorted({a["name"] for a in agents if a["name"]}, key=str.lower)
print(f"{len(names)} different names:\n")
for n in names:
    print("•", n)

## Step 7️⃣ Find MY robot 🎯

Everybody's robots are mixed on the shelf. How do we pick yours?
Every robot remembers who created it. So we:
1. Ask the room "Who am I?" (WhoAmI) and get our secret number.
2. Ask the shelf "Show me only robots created by that number."

We only check the rooms we could open (fast!). You can also type part of a name in NAME_FILTER, for example "Classification".

In [ ]:
NAME_FILTER = None          # for example "Classification" to search by name

open_rooms = [e for e in envs if e["name"] in {a["room"] for a in agents}]
fresh = {e["url"]: get_token(e["url"] + "/.default") for e in open_rooms}    # fresh stickers
save_cache()

def my_robots_in(env):
    base = f"{env['url']}/api/data/v9.2"
    headers = {"Authorization": f"Bearer {fresh[env['url']]}", "Accept": "application/json"}
    try:
        who = requests.get(f"{base}/WhoAmI", headers=headers, timeout=30)
        who.raise_for_status()
        my_id = who.json()["UserId"]
        r = requests.get(
            f"{base}/bots?$select={BOT_SELECT}&$filter=_createdby_value eq {my_id}&$orderby=createdon desc",
            headers=headers, timeout=30)
        r.raise_for_status()
    except Exception as exc:
        print("  (could not check", env["name"], "->", exc, ")")
        return []
    bots = r.json()["value"]
    if NAME_FILTER:
        bots = [b for b in bots if NAME_FILTER.lower() in (b.get("name") or "").lower()]
    return [{"room": env["name"], "room_url": env["url"], **b} for b in bots]

with ThreadPoolExecutor(max_workers=8) as pool:
    mine = [b for found in pool.map(my_robots_in, open_rooms) for b in found]
mine.sort(key=lambda b: b["createdon"], reverse=True)

print(f"\n🎉 Robots created by {USER}: {len(mine)}\n")
for b in mine:
    print(f"Name         : {b['name']}")
    print(f"Agent ID     : {b['botid']}")
    print(f"Room         : {b['room']}")
    print(f"State        : {'Active' if b.get('statecode') == 0 else 'Inactive'}")
    print(f"Shared?      : {'Yes, published ' + b['publishedon'] if b.get('publishedon') else 'No, still a draft'}")
    print(f"Created      : {b['createdon']}")
    print(f"Last change  : {b['modifiedon']}\n")

## Step 8️⃣ Save the lists to files 💾

Two files appear next to this notebook:
* **agent_names.txt** — just the names (easy to read)
* **agents.json** — all the details

In [ ]:
with open("agent_names.txt", "w") as f:
    for a in sorted(agents, key=lambda x: (x["room"].lower(), (x["name"] or "").lower())):
        f.write(f"{a['room']}\t{a['name']}\n")

with open("agents.json", "w") as f:
    json.dump({"agents": agents, "locked_rooms": locked_rooms}, f, indent=2)

print("Saved: agent_names.txt and agents.json 🟩")

## 🆘 Something went wrong?

| What you see | What it means | What to do |
| :--- | :--- | :--- |
| **No module named msal** | Tool not installed | Remove # in Step 0 and run it once |
| **Browser did not open** | Computer can't open a window | Tell me, we can use a "device code" instead |
| **AADSTS... sign-in error** | Your company blocks this door | Ask IT, or register your own app (I can help) |
| **Many rooms are 🔒 (403)** | You have no key for them | Normal. Ask the room's owner for access |
| **0 robots created by you** | Your robot is in a locked room, or the sticker is old | Run Step 7 again; check NAME_FILTER is empty |
| **Want to sign in again from scratch** | | Delete the file `.token_cache.json` |

## 🧠 One sentence to remember

**Sign in -> ask which rooms exist -> read each room's shelf (bots) -> print robot names.**